# فصل 8: یادگیری عمیق در GeoAI

کد این نوت‌بوک عیناً از فصل 8 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۸-۴) کارگاه: MLP برای داده‌های جدولی مکانی


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("/content/drive/MyDrive/GeoAI/landslide.csv")
cols = ["slope", "aspect", "rainfall", "ndvi",
        "dist_fault", "dist_road", "curvature"]

X = df[cols].values             # 7 continuous numeric features
y = df["landslide"].values      # 0 = safe, 1 = landslide

X = StandardScaler().fit_transform(X)   # normalise: vital for neural nets

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = Sequential([
    Dense(64, activation="relu", input_shape=(7,)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),     # probability of landslide
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

es = EarlyStopping(monitor="val_loss", patience=5,
                   restore_best_weights=True)

history = model.fit(X_tr, y_tr, validation_split=0.2,
                    epochs=100, batch_size=32, callbacks=[es])

## ۸-۵) شبکه‌های پیچشی (CNN): چشم هوش مصنوعی


In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential

model = Sequential([
    Conv2D(64, (3, 3), activation="relu",
           input_shape=(128, 128, 4)),    # 64 kernels
    MaxPooling2D((2, 2)),                 # halve the feature map
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid"),       # probability of a building
])

## ۸-۶) شبکه‌های بازگشتی (LSTM): حافظه‌ی هوش مصنوعی


In [ ]:
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.models import Sequential

# input shape: (samples, time_steps, features)
#   e.g. 60 months of one variable -> (samples, 60, 1)
model = Sequential([
    LSTM(50, input_shape=(60, 1)),   # 50 memory units
    Dense(1),                        # predict the next value
])

model.compile(optimizer="adam", loss="mse")   # mse for a numeric target